# Purchase-price coverage

What share of products have a purchase price (`inköpspris`)?

A **product** is one `product_code` in the latest stock snapshot, `data/nedladdningar/stock_report_2026-11-02.csv`. The same product listed in several stores counts once.

A product **has a purchase price** when that code appears in the order-line history, `data/nedladdningar/product_sales_items_2024-02-12_to_2026-11-02.csv`, and the latest `unit_price` is greater than zero. That is the field the order lists store as `senast_kända_inköpspris`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

STOCK_PATH = Path("data/nedladdningar/stock_report_2026-11-02.csv")
ITEMS_PATH = Path("data/nedladdningar/product_sales_items_2024-02-12_to_2026-11-02.csv")


def normalise_product_code(code):
    """Same key as scripts/_common._normalise_product_code."""
    if code is None or (isinstance(code, float) and pd.isna(code)):
        return None
    text = str(code).strip()
    if not text or text.lower() == "nan":
        return None
    try:
        number = float(text)
    except (TypeError, ValueError, OverflowError):
        return text
    if number != number:
        return None
    as_int = int(number)
    if float(as_int) == number:
        return str(as_int)
    return text


stock = pd.read_csv(STOCK_PATH)
items = pd.read_csv(
    ITEMS_PATH,
    usecols=["created_at", "product_code", "product_name", "supplier_name", "unit_price"],
)

stock["code"] = stock["product_code"].map(normalise_product_code)
items["code"] = items["product_code"].map(normalise_product_code)
items["ordered_at"] = pd.to_datetime(items["created_at"], errors="coerce")
items["unit_price"] = pd.to_numeric(items["unit_price"], errors="coerce")

print(f"Stock rows: {len(stock):,} across {stock['store_name'].nunique()} stores")
print(f"Unique product codes in stock: {stock['code'].nunique():,}")
print(f"Order lines: {len(items):,}")
print(f"Order lines with unit_price > 0: {(items['unit_price'] > 0).sum():,} / {len(items):,}")
print(f"Order lines with no product code: {items['code'].isna().sum():,}")
print(f"Unit price min / median / max: {items['unit_price'].min():.2f} / {items['unit_price'].median():.2f} / {items['unit_price'].max():.2f}")


## What the files contain

The stock file is the assortment on 2026-11-02: 5,229 store listings and 829 distinct product codes, across 10 stores.

The items file is every order line from 2024-02-12 through 2026-11-02 (226,992 lines). Every line already has a unit price above zero, from 0.10 to 79.00. A product can therefore lack a purchase price only when its code never appears on an order line. 8,511 lines have a price but no product code, so those prices cannot be attached to a product.


In [ ]:
latest_price = (
    items.dropna(subset=["code"])
    .sort_values("ordered_at")
    .drop_duplicates("code", keep="last")
)
price_by_code = latest_price.loc[latest_price["unit_price"] > 0, ["code", "unit_price", "product_name", "supplier_name"]]
price_by_code = price_by_code.drop_duplicates("code").set_index("code")

products = (
    stock.dropna(subset=["code"])
    .sort_values(["code", "stock"], ascending=[True, False])
    .drop_duplicates("code", keep="first")
    .copy()
)
products["purchase_price"] = products["code"].map(price_by_code["unit_price"])
products["has_purchase_price"] = products["purchase_price"].notna()
products["max_stock"] = stock.groupby("code")["stock"].max().reindex(products["code"]).to_numpy()
products["n_stores"] = stock.groupby("code")["store_name"].nunique().reindex(products["code"]).to_numpy()
products["active"] = stock.groupby("code")["product_status"].max().reindex(products["code"]).to_numpy() == 1

n = len(products)
n_with = int(products["has_purchase_price"].sum())
n_without = n - n_with

print(f"Products: {n}")
print(f"With a purchase price: {n_with} ({n_with / n:.1%})")
print(f"Without a purchase price: {n_without} ({n_without / n:.1%})")

in_stock = products["max_stock"] > 0
active = products["active"]
print(
    f"Stock above zero in at least one store: "
    f"{int(products.loc[in_stock, 'has_purchase_price'].sum())} / {int(in_stock.sum())} "
    f"({products.loc[in_stock, 'has_purchase_price'].mean():.1%})"
)
print(
    f"Active (status 1 in at least one store): "
    f"{int(products.loc[active, 'has_purchase_price'].sum())} / {int(active.sum())} "
    f"({products.loc[active, 'has_purchase_price'].mean():.1%})"
)
print(
    f"Inactive: "
    f"{int(products.loc[~active, 'has_purchase_price'].sum())} / {int((~active).sum())} "
    f"({products.loc[~active, 'has_purchase_price'].mean():.1%})"
)

item_codes = set(price_by_code.index)
stock_codes = set(products["code"])
print(f"Order-line codes with a price: {len(item_codes):,}")
print(f"Still in the stock snapshot: {len(item_codes & stock_codes):,}")
print(f"Priced codes no longer in stock: {len(item_codes - stock_codes):,}")


## Coverage

**803 of 829 products (96.9%) have a purchase price.** The other 26 (3.1%) never appear on an order line in this extract, so there is no unit price to carry forward.

The gap is not an inactive-product problem. Products with stock above zero are covered at 726 of 746 (97.3%). Active products are at 678 of 701 (96.7%), and inactive products at 125 of 128 (97.7%).

Of the 1,021 product codes that have a price on an order line, 803 are still in the snapshot and 218 have dropped out of the assortment. Price coverage of the order history itself is complete: every coded order line has a positive unit price.


In [ ]:
store_rows = stock.dropna(subset=["code"]).copy()
store_rows["has_purchase_price"] = store_rows["code"].isin(price_by_code.index)
by_store = (
    store_rows.groupby("store_name", as_index=False)
    .agg(with_price=("has_purchase_price", "sum"), listings=("has_purchase_price", "count"))
    .sort_values("listings", ascending=True)
)
by_store["share"] = by_store["with_price"] / by_store["listings"]

fig, axes = plt.subplots(1, 2, figsize=(12.5, 6), gridspec_kw={"width_ratios": [1, 1.7]})

axes[0].bar(
    ["Has purchase price", "Missing"],
    [n_with, n_without],
    color=["#2a6f4e", "#c4554a"],
)
axes[0].set_ylabel("Products")
axes[0].set_title(f"{n_with / n:.1%} of products have a purchase price")
for index, value in enumerate([n_with, n_without]):
    axes[0].text(index, value + 12, str(value), ha="center", va="bottom")
axes[0].set_ylim(0, n_with * 1.15)

axes[1].barh(by_store["store_name"], by_store["share"] * 100, color="#3d6f9a")
axes[1].set_xlabel("Share of store listings with a purchase price (%)")
axes[1].set_xlim(94, 103)
axes[1].set_title("Coverage by store")
for y_pos, row in enumerate(by_store.itertuples(index=False)):
    axes[1].text(
        row.share * 100 + 0.12,
        y_pos,
        f"{row.share:.1%}  (n={int(row.listings)})",
        va="center",
        fontsize=8,
    )

fig.tight_layout()
plt.show()

print((by_store.assign(share=lambda d: (d["share"] * 100).round(1))
       .sort_values("share"))
      .to_string(index=False))


## Stores

Store listings are slightly better covered than unique products: 5,146 of 5,229 listings (98.4%). The 26 unpriced products account for 83 listings, about 3.2 stores each, against about 6.3 stores for a typical product, so they weigh less here than in the product count.

No store looks like a broken feed. Lager kållby (654 of 671, 97.5%) and Torp butik (740 of 759, 97.5%) are the lowest. Sandsund butik (597 of 602, 99.2%) and Bosund GAMLA STÄNGD (475 of 479, 99.2%) are the highest.


In [ ]:
missing = products.loc[
    ~products["has_purchase_price"],
    ["code", "product_name", "active", "max_stock", "n_stores"],
].sort_values(["active", "max_stock", "product_name"], ascending=[False, False, True])
missing = missing.rename(columns={
    "code": "product_code",
    "active": "active_in_a_store",
    "max_stock": "highest_stock",
    "n_stores": "stores",
})
print(f"{len(missing)} products have no purchase price")
print(f"Of those, {(missing['highest_stock'] > 0).sum()} have stock above zero")
print(f"Of those, {int(missing['active_in_a_store'].sum())} are active in at least one store")
missing


## Products with no purchase price

The 26 codes are specific SKUs, not a whole supplier. 20 of them have stock above zero, and 23 are active, so most of them can still be ordered.

They include Tajma meat cuts on short internal codes (2014, 3261, 733074, 733076), the RW car-care line, and single grocery SKUs such as Colgate Total, Elovena havre-lingonbär, Felix tomat-chili, Havu tallsåpa, and Tolu tall. Three are inactive and at zero stock: Närpes Memmagräddglass, Trocadero 33cl, and Träskulla probeef nöt maletkött frusen.

A few names sit next to a different SKU that does have a price (Elovena banana-chocolate versus havre-lingonbär, other Trocadero lines, other Tajma ytterfile cuts). Those sibling prices are not this product's purchase price, because the price is stored per product code and these codes have no order line of their own.


## Summary

### Q&A

**96.9% of products have a purchase price: 803 of 829 product codes in the stock snapshot of 2026-11-02.** The remaining 26 codes (3.1%) have no `inköpspris` because they never appear in the order-line history from 2024-02-12 to 2026-11-02. Every order line that has a product code also has a positive unit price, so the gap is missing history rather than a blank price field.

### Data Analysis Key Findings

- All 226,992 order lines have a unit price between 0.10 and 79.00. 8,511 lines have a price but no product code, so that price cannot be joined to a product.
- 1,021 order-line codes have a price. 803 are still in stock and 218 have left the assortment.
- Coverage stays high on the shelf: 726 of 746 products with stock above zero (97.3%), 678 of 701 active products (96.7%), and 125 of 128 inactive products (97.7%).
- Store listings are at 5,146 of 5,229 (98.4%). Lager kållby and Torp butik are lowest at 97.5%, and Sandsund butik is highest at 99.2%.
- Of the 26 unpriced products, 20 still have stock and 23 are active. They are individual SKUs, including Tajma cuts on internal codes and the RW car-care line.

### Insights or Next Steps

- Fill purchase prices for the 20 in-stock products before the next order run. Those are the codes that will show an empty `senast_kända_inköpspris`.
- Do not copy a sibling SKU's price onto these codes. A few names look related, but each product code needs its own order line or a manually entered price.
